# Pakistan News Intelligence Research
### Automated NLP Analysis of 350,000 Headlines from Dawn News
This notebook presents a visual analysis of news data focused on Pakistan's domestic policy, regional security, and socio-economic trends.

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import os
from collections import Counter

if os.getcwd().endswith('notebooks'):
    os.chdir('..')

df = pd.read_parquet('data/processed/processed_nlp_features.parquet')
df['published_at'] = pd.to_datetime(df['published_at'])

print(f"Successfully loaded {len(df):,} Pakistan-centric headlines.")
display(df.head(3))

Successfully loaded 350,718 Pakistan-centric headlines.


,id,published_at,headline,category,body,url,extracted_entities,sentiment_score,sentiment_label,dominant_topic
0,1,2010-01-01,Calendar antics,Blogs,Today I woke up to the fact that this was in f...,https://www.dawn.com/news/813107/new-decade-ne...,[],0.0000,neutral,13
1,2,2010-01-01,Selection woes,Blogs,One hundred and seventy runs was the margin of...,https://www.dawn.com/news/813106/selection-woes,[],-0.4404,negative,14
2,3,2010-01-01,2009: The year in quotes,Blogs,January: “Why would she leave it to me if she ...,https://www.dawn.com/news/813105/2009-the-year...,[],0.0000,neutral,13


## 1. Dataset Overview
A high-level summary of the dataset volume and the overall emotional baseline of Pakistan's media landscape calculated via VADER sentiment analysis.

In [2]:
total_headlines = len(df)
avg_sentiment = df['sentiment_score'].mean()
unique_topics = df['dominant_topic'].nunique()

print(f"Total Headlines: {total_headlines:,}")
print(f"National Avg Sentiment: {avg_sentiment:.3f}")
print(f"Total Unique Topics: {unique_topics}")

Total Headlines: 350,718
National Avg Sentiment: -0.085
Total Unique Topics: 20


## 2. Topic Distribution
This chart displays the volume of news within each numerical topic ID. It highlights the concentration of reporting across the modeled categories.

In [3]:
topic_data = df['dominant_topic'].value_counts().reset_index()
topic_data.columns = ['Topic_ID', 'Count']
topic_data['Topic_ID'] = topic_data['Topic_ID'].astype(str)

fig_topics = px.bar(
    topic_data, 
    x='Count', 
    y='Topic_ID', 
    orientation='h',
    title='News Volume by Topic ID',
    color='Count',
    color_continuous_scale='RdBu_r'
)
fig_topics.update_layout(yaxis={'categoryorder':'total ascending'}, height=600)
fig_topics.show()

## 3. Sentiment Pulse
A 7-day rolling average to identify major shifts in the national mood. Dips often correspond to periods of political instability, security incidents, or economic inflation data.

In [4]:
daily_series = df.groupby(df['published_at'].dt.date)['sentiment_score'].mean().reset_index()
daily_series['rolling_avg'] = daily_series['sentiment_score'].rolling(window=7).mean()

fig_trend = px.line(
    daily_series, 
    x='published_at', 
    y='rolling_avg',
    title='National News Sentiment Trend (7-Day Rolling)',
    labels={'rolling_avg': 'Sentiment Score', 'published_at': 'Date'}
)
fig_trend.add_hline(y=0, line_dash="dash", line_color="black", opacity=0.5)
fig_trend.show()

## 4. Key Actors and Locations (NER)
Identifying the most prominent political figures, organizations, and cities mentioned in Dawn News headlines.

In [5]:
all_ents = [ent for sublist in df['extracted_entities'].dropna() for ent in sublist]
ent_counts = Counter(all_ents).most_common(15)
ent_df = pd.DataFrame(ent_counts, columns=['Entity', 'Mentions'])

fig_ents = px.pie(
    ent_df, 
    values='Mentions', 
    names='Entity', 
    title='Share of Voice: Prominent Entities in Pakistan News',
    hole=0.4
)
fig_ents.show()

## 5. Topic Specific Content Analysis
Examining specific samples from the dominant topics to interpret the numerical clusters and verify sentiment scoring.

In [6]:
target_id = df['dominant_topic'].unique()[0]
st_sample = df[df['dominant_topic'] == target_id].sort_values(by='sentiment_score')

print(f"Representative Headlines for Topic ID: {target_id}")
display(st_sample[['published_at', 'headline', 'sentiment_score']].head(3))
display(st_sample[['published_at', 'headline', 'sentiment_score']].tail(3))

Representative Headlines for Topic ID: 13


,published_at,headline,sentiment_score
145627,2017-03-12,UN says world faces worst humanitarian crisis ...,-0.9201
318653,2024-09-23,"As wars rage, UN’s critics say global body is ...",-0.9136
319944,2024-10-12,"Trump ratchets up rhetoric, calls for death pe...",-0.9118


,published_at,headline,sentiment_score
344990,2025-10-13,Trio win Nobel economics prize for work on inn...,0.9062
316109,2024-08-14,"With unwavering hope and calls for peace, cele...",0.9153
257199,2021-12-25,'Happy holidays': Pakistani celebrities wish a...,0.9312
